# Qwen SQL Inference Notebook

This notebook is designed for Google Colab and supports these experiment modes:

- zero-shot + thinking
- zero-shot + non-thinking
- zero-shot + thinking + outlines
- zero-shot + non-thinking + outlines

The output JSON always contains these six fields:

- `instance_id`
- `question`
- `gold_sql`
- `predicted_sql`
- `latency_ms`
- `num_tokens_generated`

In [ ]:
!pip install -q transformers accelerate sentencepiece
# !pip install -q outlines
!pip install -q xgrammar

In [ ]:
import os
import re
import json
import time
from typing import Dict, Any, List, Optional

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# =========================
# User config
# =========================

INPUT_PATH = "inputs/wikisql_train_sample50.json"   # Change this to your uploaded JSON path
OUTPUT_DIR = "outputs"

MODEL_NAME = "Qwen/Qwen3-0.6B"

# Options:
# MODE = "thinking" or "non-thinking"
# CONSTRAINT_METHOD = "none" or "outlines" or "xgrammar"
# TRAIN_MODE = "zero" or "ft"
DATASET_NAME = "wikisql"
MODE = "non-thinking"
CONSTRAINT_METHOD = "xgrammar"
TRAIN_MODE = "zero"

MAX_NEW_TOKENS = 128

# For SQL generation, deterministic decoding is usually more stable.
TEMPERATURE = None
TOP_P = None
TOP_K = None
MIN_P = None

TRUST_REMOTE_CODE = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=TRUST_REMOTE_CODE,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=TRUST_REMOTE_CODE,
    torch_dtype="auto",
    device_map="auto",
)

model.eval()
print("Model loaded.")
print("Device:", DEVICE)

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} examples.")
if len(data) > 0:
    print("Sample keys:", list(data[0].keys()))

def get_default_sampling(mode: str) -> Dict[str, Any]:
    """
    Use deterministic decoding for more stable SQL generation.
    """
    return {
        "do_sample": False,
    }


def build_generation_kwargs(
    mode: str,
    max_new_tokens: int,
    temperature: Optional[float],
    top_p: Optional[float],
    top_k: Optional[int],
    min_p: Optional[float],
) -> Dict[str, Any]:
    kwargs = get_default_sampling(mode)
    kwargs["max_new_tokens"] = max_new_tokens

    if kwargs.get("do_sample", False):
        if temperature is not None:
            kwargs["temperature"] = temperature
        if top_p is not None:
            kwargs["top_p"] = top_p
        if top_k is not None:
            kwargs["top_k"] = top_k
        if min_p is not None:
            kwargs["min_p"] = min_p

    return kwargs


def format_schema_text(schema: Dict[str, Any]) -> str:
    """
    Convert schema metadata into a simple text format that is easy for the model to read.
    """
    if not schema:
        return "No schema provided."

    table_names = schema.get("table_names_original") or schema.get("table_names") or []
    column_names = schema.get("column_names_original") or schema.get("column_names") or []
    column_types = schema.get("column_types") or ["unknown"] * len(column_names)

    lines = []

    for table_id, table_name in enumerate(table_names):
        lines.append(f"TABLE: {table_name}")
        lines.append("COLUMNS:")
        for idx, pair in enumerate(column_names):
            if not isinstance(pair, (list, tuple)) or len(pair) != 2:
                continue
            tid, col_name = pair
            if tid != table_id:
                continue
            if col_name == "*":
                continue

            col_type = column_types[idx] if idx < len(column_types) else "unknown"
            lines.append(f"- {col_name} [{col_type}]")
        lines.append("")

    text = "\n".join(lines).strip()
    return text if text else "No schema provided."


def build_plain_prompt(example: Dict[str, Any]) -> str:
    """
    Build a stronger prompt for text-to-SQL generation.
    """
    instance_id = example.get("instance_id", "")
    question = example.get("question", "")
    schema_text = format_schema_text(example.get("schema", {}))

    prompt = f"""You are a text-to-SQL system.

Task:
Convert the question into exactly one SQL query for the given schema.

Rules:
1. Output exactly one SQL query and nothing else.
2. The query must start with SELECT.
3. Use only the table name: table
4. Use only columns that appear in the schema.
5. Preserve string values exactly from the question when possible.
6. Put text values in single quotes.
7. Do not invent columns, values, or conditions.
8. Do not add explanation.
9. Prefer the simplest correct WikiSQL-style query.

Instance ID: {instance_id}

Schema:
{schema_text}

Question:
{question}

SQL:
"""
    return prompt


def build_qwen_prompt(tokenizer, example: Dict[str, Any], mode: str) -> str:
    """
    Build a chat-formatted prompt for Qwen.
    """
    user_prompt = build_plain_prompt(example)

    messages = [
        {"role": "system", "content": "You are a precise text-to-SQL assistant."},
        {"role": "user", "content": user_prompt},
    ]

    enable_thinking = (mode == "thinking")

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    return prompt


# def clean_sql(text: str, mode: str = "non-thinking") -> str:
#     """
#     Extract final SQL from model output.

#     In thinking mode, keep only the text after </think>.
#     Then remove markdown fences and keep the SQL starting from SELECT.
#     """
#     if text is None:
#         return ""

#     text = text.strip()

#     # Handle thinking mode
#     if mode == "thinking":
#         if "</think>" in text:
#             text = text.split("</think>", 1)[1].strip()
#         elif "select" in text.lower():
#             text = text[text.lower().rfind("select"):].strip()

#     # Remove markdown code fences
#     text = text.replace("```sql", "").replace("```", "").strip()

#     # Normalize whitespace
#     text = text.replace("\n", " ")
#     text = " ".join(text.split())

#     # Keep only from the first SELECT
#     lower_text = text.lower()
#     if "select" in lower_text:
#         text = text[lower_text.find("select"):].strip()
#     else:
#         return ""

#     if not text.lower().startswith("select"):
#         return ""

#     if text and not text.endswith(";"):
#         text += ";"

#     return text
# def clean_sql(text: str, mode: str = "non-thinking") -> str:
#     """
#     Extract clean SQL from model output.

#     - In thinking mode: keep only text after </think>
#     - Remove special tokens like <|im_end|>
#     - Remove markdown fences
#     - Keep only SQL starting from SELECT
#     - Ensure single trailing semicolon
#     """

#     if text is None:
#         return ""

#     text = text.strip()

#     # --------------------------------------------------
#     # Step 1: Handle thinking mode
#     # --------------------------------------------------
#     if mode == "thinking":
#         if "</think>" in text:
#             text = text.split("</think>", 1)[1].strip()
#         else:
#             # fallback: take last SELECT
#             lower = text.lower()
#             if "select" in lower:
#                 text = text[lower.rfind("select"):].strip()

#     # --------------------------------------------------
#     # Step 2: Remove special tokens (VERY IMPORTANT)
#     # --------------------------------------------------
#     # Remove Qwen / chat template artifacts
#     text = re.sub(r"<\|.*?\|>", "", text)

#     # --------------------------------------------------
#     # Step 3: Remove markdown code blocks
#     # --------------------------------------------------
#     text = text.replace("```sql", "").replace("```", "").strip()

#     # --------------------------------------------------
#     # Step 4: Normalize whitespace
#     # --------------------------------------------------
#     text = text.replace("\n", " ")
#     text = " ".join(text.split())

#     # --------------------------------------------------
#     # Step 5: Keep only SQL starting from SELECT
#     # --------------------------------------------------
#     lower_text = text.lower()
#     if "select" in lower_text:
#         text = text[lower_text.find("select"):].strip()
#     else:
#         return ""

#     # --------------------------------------------------
#     # Step 6: Remove trailing garbage after semicolon
#     # --------------------------------------------------
#     if ";" in text:
#         # text = text.split(";", 1)[0] + ";"
#         text = text.split(";", 1)[0]

#     # --------------------------------------------------
#     # Step 7: Final safety
#     # --------------------------------------------------
#     if not text.lower().startswith("select"):
#         return ""

#     return text
def clean_sql(text: str, mode: str = "non-thinking") -> str:
    if text is None:
        return ""

    text = text.strip()

    if mode == "thinking":
        if "</think>" in text:
            text = text.split("</think>", 1)[1].strip()
        elif "select" in text.lower():
            text = text[text.lower().rfind("select"):].strip()

    text = re.sub(r"<\|.*?\|>", "", text)
    text = text.replace("```sql", "").replace("```", "").strip()
    text = text.replace("\n", " ")
    text = " ".join(text.split())

    lower_text = text.lower()
    if "select" not in lower_text:
        return ""

    text = text[lower_text.find("select"):].strip()

    if ";" in text:
        text = text.split(";", 1)[0].strip()

    # Reject obviously incomplete SQL
    bad_suffixes = ("where", "and", "or", "(", "=", ">", "<", ">=", "<=", "!=")
    if text.lower().endswith(bad_suffixes):
        return ""
    if text.count("(") != text.count(")"):
        return ""

    if not text.lower().startswith("select"):
        return ""

    return text


def validate_output_record(record: Dict[str, Any]) -> None:
    """
    Ensure required keys are included in the output record.
    """
    required_keys = [
        "instance_id",
        "question",
        "gold_sql",
        "predicted_sql",
        "latency_ms",
        "num_tokens_generated",
    ]

    missing = [k for k in required_keys if k not in record]
    if missing:
        raise ValueError(f"Missing required keys: {missing}")


def generate_plain(
    model,
    tokenizer,
    prompt: str,
    generation_kwargs: Dict[str, Any],
):
    """
    Plain Hugging Face generation without constrained decoding.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    start_time = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **generation_kwargs,
        )

    latency_ms = (time.perf_counter() - start_time) * 1000.0

    generated_ids = outputs[0][input_len:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=False)
    num_tokens_generated = int(generated_ids.shape[0])

    return text, num_tokens_generated, latency_ms


def build_simple_sql_regex() -> str:
    """
    A minimal regex constraint.
    This is intentionally simple because overly strict regex often hurts performance.
    """
    return r"SELECT\s+.+?\s+FROM\s+table(?:\s+WHERE\s+.+?)?\s*;?"

def build_better_wikisql_regex() -> str:
    ident = r"[A-Za-z_][A-Za-z0-9_ ]*"
    number = r"-?\d+(?:\.\d+)?"
    string = r"'[^']*'"
    value = rf"(?:{string}|{number})"
    op = r"(?:=|!=|>|<|>=|<=)"
    agg = rf"(?:COUNT\(\*\)|COUNT\({ident}\)|MAX\({ident}\)|MIN\({ident}\)|SUM\({ident}\)|AVG\({ident}\))"
    select_expr = rf"(?:\*|{ident}|{agg})"
    condition = rf"{ident}\s+{op}\s+{value}"

    # Allow 0 to 2 WHERE conditions, but each condition must be complete.
    return (
        rf"SELECT\s+{select_expr}\s+FROM\s+table"
        rf"(?:\s+WHERE\s+{condition}"
        rf"(?:\s+(?:AND|OR)\s+{condition}){{0,1}})?"
        rf"\s*;?"
    )

class OutlinesRunner:
    """
    A version-compatible Outlines wrapper.
    """
    def __init__(self, model, tokenizer):
        import outlines
        from outlines.types import Regex

        self.outlines = outlines
        self.Regex = Regex
        self.tokenizer = tokenizer
        self.outlines_model = outlines.from_transformers(model, tokenizer)

    def generate_regex(
        self,
        prompt: str,
        regex_pattern: str,
        max_new_tokens: int = 128,
    ):
        start_time = time.perf_counter()

        output = self.outlines_model(
            prompt,
            self.Regex(regex_pattern),
            max_new_tokens=max_new_tokens,
        )

        latency_ms = (time.perf_counter() - start_time) * 1000.0

        text = output if isinstance(output, str) else str(output)
        num_tokens_generated = len(
            self.tokenizer.encode(text, add_special_tokens=False)
        )

        return text, num_tokens_generated, latency_ms


def build_output_filename(
    dataset_name: str,
    mode: str,
    train_mode: str,
    constraint_method: str,
) -> str:
    """
    Required naming convention:
    result_{dataset name}_{thinking mode}_{zero/ft}_{constrained decoding methods}.json
    """
    return f"result_{dataset_name}_{mode}_{train_mode}_{constraint_method}.json"



Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded.
Device: cuda
Loaded 50 examples.
Sample keys: ['instance_id', 'db', 'question', 'schema', 'gold_sql_query']


# XGrammer

In [ ]:
import xgrammar as xgr
def escape_ebnf_literal(s: str) -> str:
    # Use JSON string quoting so spaces / punctuation are handled safely
    return json.dumps(s)

def build_wikisql_ebnf(example: dict) -> str:
    schema = example["schema"]
    columns = [c[1] for c in schema["column_names_original"]]
    col_rules = " | ".join(escape_ebnf_literal(c) for c in columns)

    grammar = f"""
root ::= ws select_stmt ws

select_stmt ::= "SELECT" ws distinct_opt ws select_item ws "FROM" ws table_name ws where_opt ws order_opt ws limit_opt

distinct_opt ::= "" | "DISTINCT" ws

select_item ::= "*" | column_name | agg_func ws "(" ws "*" ws ")" | agg_func ws "(" ws column_name ws ")"
agg_func ::= "COUNT" | "MAX" | "MIN" | "SUM" | "AVG"

where_opt ::= "" | "WHERE" ws condition
condition ::= column_name ws op ws value

op ::= "=" | "!=" | "<" | ">" | "<=" | ">=" | "LIKE"

order_opt ::= "" | "ORDER" ws "BY" ws column_name ws order_dir
order_dir ::= "" | "ASC" | "DESC"

limit_opt ::= "" | "LIMIT" ws number

table_name ::= {escape_ebnf_literal("table")}
column_name ::= {col_rules}

value ::= quoted_string | number | bare_value
quoted_string ::= "'" quoted_body* "'"
quoted_body ::= [^'\\\\\\x00-\\x1F]
number ::= ["-"]? [0-9]+ ("." [0-9]+)?
bare_value ::= bare_char+
bare_char ::= [A-Za-z0-9_./()\\-]
ws ::= [ \\t\\n]*
"""
    print("build_wikisql_ebnf_1")
    return grammar.strip()


def build_wikisql_ebnf_new(example: dict) -> str:
    schema = example["schema"]
    columns = [c[1] for c in schema["column_names_original"]]
    col_rules = " | ".join(escape_ebnf_literal(c) for c in columns)

    grammar = f"""
root ::= select_stmt
select_stmt ::= "SELECT" ws select_item ws "FROM" ws table_name where_opt order_opt limit_opt

select_item ::= "*"
              | agg_func ws "(" ws "*" ws ")"
              | agg_func ws "(" ws column_name ws ")"
              | column_name

agg_func ::= "COUNT" | "MAX" | "MIN" | "SUM" | "AVG"

# 支持多条件 AND/OR
where_opt ::= "" | ws "WHERE" ws condition (ws ("AND" | "OR") ws condition)*
condition ::= column_name ws op ws value

op ::= "=" | "!=" | "<" | ">" | "<=" | ">=" | "LIKE"

order_opt ::= "" | ws "ORDER" ws "BY" ws column_name (ws ("ASC" | "DESC"))?
limit_opt ::= "" | ws "LIMIT" ws [0-9]+

table_name ::= "table"
column_name ::= {col_rules}

# value 放宽，不要过度约束
value ::= quoted_string | number | bare_word
quoted_string ::= "\'" [^']* "\'"
number ::= "-"? [0-9]+ ("." [0-9]+)?
bare_word ::= [A-Za-z0-9_]+

ws ::= [ \\t]*
"""
    return grammar.strip()

### extend ebnf

In [ ]:
import json
import xgrammar as xgr

def escape_ebnf_literal(s: str) -> str:
    escaped = s.replace("\\", "\\\\").replace('"', '\\"')
    return f'"{escaped}"'

# def escape_ebnf_literal(s: str) -> str:
#     return json.dumps(s, ensure_ascii=False)


def build_sql_ebnf_extend(
    columns: list[str],
    table: str,
    *,
    allow_or: bool = True,
    allow_subquery: bool = False,
    allow_join: bool = False,
    multi_table: list[str] | None = None,
) -> str:
    """
    通用 NL2SQL grammar，schema 信息动态注入。

    Parameters
    ----------
    columns      : 当前 schema 的列名列表
    table        : 主表名（单表场景）
    allow_or     : WHERE 是否允许 OR（WikiSQL 不需要，Spider 需要）
    allow_join   : 是否允许 JOIN（多表场景）
    multi_table  : JOIN 场景下的所有表名，None 则只用 table
    """
    col_rules   = " | ".join(escape_ebnf_literal(c) for c in columns if c != "*")
    tables      = multi_table if multi_table else [table]
    table_rules = " | ".join(escape_ebnf_literal(t) for t in tables)

    # WHERE
    and_or = '("AND" | "OR")' if allow_or else '"AND"'

    # JOIN
    if allow_join and multi_table:
        join_clause = r"""
join_clause  ::= "" | (ws join_type ws "JOIN" ws table_name ws "ON" ws column_name ws "=" ws column_name)+
join_type    ::= "" | "INNER" | "LEFT" | "RIGHT" | "FULL OUTER"
"""
        join_ref = " join_clause"
    else:
        join_clause = ""
        join_ref    = ""

    grammar = rf"""
root         ::= select_stmt

select_stmt  ::= "SELECT" ws select_expr ws "FROM" ws table_name{join_ref} where_clause order_clause limit_clause

# ── SELECT ───────────────────────────────────────────────────
select_expr  ::= agg_expr | column_name | "*"
agg_expr     ::= agg_func ws "(" ws ("*" | column_name) ws ")"
agg_func     ::= "COUNT" | "MAX" | "MIN" | "SUM" | "AVG"

# ── WHERE ────────────────────────────────────────────────────
where_clause ::= "" | ws "WHERE" ws condition (ws {and_or} ws condition)*
condition    ::= column_name ws op ws value

op           ::= "=" | "!=" | ">" | "<" | ">=" | "<=" | "LIKE" | "IN" | "NOT IN" | "IS NULL" | "IS NOT NULL"

# ── value ────────────────────────────────────────────────────
value        ::= number | quoted_string | bare_word | "NULL"
number       ::= "-"? [0-9]+ ("." [0-9]+)?
quoted_string ::= "\"" dq_char* "\"" | "'" sq_char* "'"
dq_char      ::= [^"\\] | "\\" ["\\/bfnrt]
sq_char      ::= [^'\\] | "\\" ['\\/bfnrt]
bare_word    ::= [A-Za-z0-9_.%+\-]+

# ── ORDER BY ─────────────────────────────────────────────────
order_clause ::= "" | ws "ORDER" ws "BY" ws column_name (ws ("ASC" | "DESC"))?

# ── LIMIT ────────────────────────────────────────────────────
limit_clause ::= "" | ws "LIMIT" ws [0-9]+

# ── JOIN（可选）──────────────────────────────────────────────{join_clause}
# ── 白名单 ───────────────────────────────────────────────────
table_name   ::= {table_rules}
column_name  ::= {col_rules}

ws           ::= [ \t]+
"""
    return grammar.strip()



In [ ]:
# ── WikiSQL ──────────────────────────────────────────────────
def from_wikisql(example: dict) -> str:
    schema  = example["schema"]
    columns = [c[1] for c in schema["column_names_original"]]
    table   = schema["table_names_original"][0]
    return build_sql_ebnf_extend(columns, table, allow_or=False, allow_join=False)

# ── Spider ───────────────────────────────────────────────────
def from_spider(example: dict) -> str:
    schema  = example["schema"]
    # Spider schema 包含多表
    columns = [c[1] for c in schema["column_names_original"] if c[1] != "*"]
    tables  = schema["table_names_original"]
    return build_sql_ebnf_extend(
        columns,
        table=tables[0],
        allow_or=True,
        allow_join=True,
        multi_table=tables,
    )


# ── custom db ──────────────────────────────────────────────
def from_custom_db(table: str, columns: list[str]) -> str:
    return build_sql_ebnf_extend(columns, table, allow_or=True, allow_join=False)

## generate XGrammar

In [ ]:
def init_xgrammar(tokenizer, model_name):
  config = AutoConfig.from_pretrained(model_name)
  tokenizer_info = xgr.TokenizerInfo.from_huggingface(tokenizer, vocab_size=config.vocab_size)
  grammar_compiler = xgr.GrammarCompiler(tokenizer_info, max_threads=8)
  return grammar_compiler

In [ ]:
def generate_xgrammar(model, tokenizer, device, prompt: str, example: dict, grammar_compiler, generation_kwargs: Dict[str, Any]):
    ebnf = build_wikisql_ebnf(example)
    compiled_grammar = grammar_compiler.compile_grammar(ebnf)
    xgr_logits_processor = xgr.contrib.hf.LogitsProcessor(compiled_grammar)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    t0 = time.time()
    outputs = model.generate(
        **inputs,
        **generation_kwargs,
        logits_processor=[xgr_logits_processor],
    )
    latency = time.time() - t0
    gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    pred_sql = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return pred_sql, int(gen_tokens.shape[0]), latency

def generate_xgrammar_new(model, tokenizer, device, prompt: str, example: dict, grammar_compiler, generation_kwargs: Dict[str, Any]):
    ebnf = build_wikisql_ebnf_new(example)
    compiled_grammar = grammar_compiler.compile_grammar(ebnf)
    xgr_logits_processor = xgr.contrib.hf.LogitsProcessor(compiled_grammar)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    t0 = time.time()
    outputs = model.generate(
        **inputs,
        **generation_kwargs,
        logits_processor=[xgr_logits_processor],
    )
    latency = time.time() - t0
    gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    pred_sql = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return pred_sql, int(gen_tokens.shape[0]), latency

In [ ]:
def generate_xgrammar_extend(model, tokenizer, device, prompt: str, example: dict, grammar_compiler, generation_kwargs: Dict[str, Any]):
    ebnf = from_wikisql(example)
    compiled_grammar = grammar_compiler.compile_grammar(ebnf)
    xgr_logits_processor = xgr.contrib.hf.LogitsProcessor(compiled_grammar)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    t0 = time.time()
    outputs = model.generate(
        **inputs,
        **generation_kwargs,
        logits_processor=[xgr_logits_processor],
    )
    latency = time.time() - t0
    gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    pred_sql = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return pred_sql, int(gen_tokens.shape[0]), latency

## run inference

### original version

In [ ]:
from transformers import AutoConfig
# Choose from: "thinking", "non-thinking"
MODE = "non-thinking"
# MODE = "thinking"

# Choose from: "zero", "ft"
TRAIN_MODE = "zero"

# Choose from: "none", "outlines" or "xgrammar"
# CONSTRAINT_METHOD = "none"
CONSTRAINT_METHOD = "xgrammar"

MAX_NEW_TOKENS = 100

os.makedirs(OUTPUT_DIR, exist_ok=True)

generation_kwargs = build_generation_kwargs(
    mode=MODE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=MIN_P,
)

outlines_runner = None
if CONSTRAINT_METHOD == "outlines":
    outlines_runner = OutlinesRunner(model, tokenizer)

grammar_compiler = None
if CONSTRAINT_METHOD == "xgrammar":
    grammar_compiler = init_xgrammar(tokenizer, MODEL_NAME)

results = []

for idx, example in enumerate(data, start=1):
    prompt = build_qwen_prompt(tokenizer, example, MODE)

    if CONSTRAINT_METHOD == "outlines":
        regex_pattern = build_better_wikisql_regex()
        raw_text, num_tokens_generated, latency_ms = outlines_runner.generate_regex(
            prompt=prompt,
            regex_pattern=regex_pattern,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    elif CONSTRAINT_METHOD == "xgrammar":
        raw_text, num_tokens_generated, latency_ms = generate_xgrammar(
        model=model,
        tokenizer=tokenizer,
        device=model.device,
        prompt=prompt,
        example=example,
        grammar_compiler=grammar_compiler,
        generation_kwargs=generation_kwargs,
    )
    else:
        raw_text, num_tokens_generated, latency_ms = generate_plain(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            generation_kwargs=generation_kwargs,
        )

    predicted_sql = clean_sql(raw_text, mode=MODE)

    # Keep all original fields from the input example.
    result = dict(example)

    # Map gold_sql_query -> gold_sql
    gold_sql_value = example.get("gold_sql_query", "")

    result.update({
        "instance_id": example.get("instance_id", ""),
        "question": example.get("question", ""),
        "gold_sql": gold_sql_value,
        "predicted_sql": predicted_sql,
        "latency_ms": round(float(latency_ms), 3),
        "num_tokens_generated": int(num_tokens_generated),
    })

    # Remove the old field if it exists
    if "gold_sql_query" in result:
        del result["gold_sql_query"]

    validate_output_record(result)
    results.append(result)

    if idx <= 3:
        print("=" * 80)
        print(f"Example {idx}")
        print("Question:", result["question"])
        print("Gold SQL:", result["gold_sql"])
        print("Predicted SQL:", result["predicted_sql"])
        print("Latency (ms):", result["latency_ms"])
        print("Generated tokens:", result["num_tokens_generated"])
    # if idx == 3:
    #   break

    if idx % 5 == 0 or idx == len(data):
        print(f"Done {idx}/{len(data)}")

output_filename = build_output_filename(
    dataset_name=DATASET_NAME,
    mode=MODE,
    train_mode=TRAIN_MODE,
    constraint_method=CONSTRAINT_METHOD,
)

output_path = os.path.join(OUTPUT_DIR, output_filename)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to:", output_path)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Example 1
Question: What is the title of the episode directed by Peter Markle and written by Jerry Stahl?
Gold SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Predicted SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' ORDER BY Written by
Latency (ms): 5.979
Generated tokens: 20
Example 2
Question: What's the total number of episodes with production code 2T6705?
Gold SQL: SELECT COUNT(Title) FROM table WHERE Production code = '2T6705'
Predicted SQL: SELECT COUNT(*) FROM table WHERE Production code = '2T6705'
Latency (ms): 1.168
Generated tokens: 18
Example 3
Question: How many times was the GFR 2006 equal to 53.2?
Gold SQL: SELECT COUNT(Whites as % of Pop.) FROM table WHERE GFR 2006 = '53.2'
Predicted SQL: SELECT COUNT(*) FROM table WHERE GFR 2006 = '53.2'
Latency (ms): 1.266
Generated tokens: 21
Done 5/50
Done 10/50
Done 15/50
Done 20/50
Done 25/50
Done 30/50
Done 35/50
Done 40/50
Done 45/50
Done 50/50
Saved to: outputs/resu

In [ ]:
# Choose from: "thinking", "non-thinking"
# MODE = "thinking"
MODE = "thinking"

# Choose from: "zero", "ft"
TRAIN_MODE = "zero"

# Choose from: "none", "outlines"，"xgrammar
# CONSTRAINT_METHOD = "none"
CONSTRAINT_METHOD = "xgrammar"

MAX_NEW_TOKENS = 1000

os.makedirs(OUTPUT_DIR, exist_ok=True)

generation_kwargs = build_generation_kwargs(
    mode=MODE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=MIN_P,
)

outlines_runner = None
if CONSTRAINT_METHOD == "outlines":
    outlines_runner = OutlinesRunner(model, tokenizer)

grammar_compiler = None
if CONSTRAINT_METHOD == "xgrammar":
    grammar_compiler = init_xgrammar(tokenizer, MODEL_NAME)

results = []

for idx, example in enumerate(data, start=1):
    prompt = build_qwen_prompt(tokenizer, example, MODE)

    if CONSTRAINT_METHOD == "outlines":
        regex_pattern = build_better_wikisql_regex()
        raw_text, num_tokens_generated, latency_ms = outlines_runner.generate_regex(
            prompt=prompt,
            regex_pattern=regex_pattern,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    elif CONSTRAINT_METHOD == "xgrammar":
        raw_text, num_tokens_generated, latency_ms = generate_xgrammar(
        model=model,
        tokenizer=tokenizer,
        device=model.device,
        prompt=prompt,
        example=example,
        grammar_compiler=grammar_compiler,
        generation_kwargs=generation_kwargs,
    )
    else:
        raw_text, num_tokens_generated, latency_ms = generate_plain(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            generation_kwargs=generation_kwargs,
        )

    predicted_sql = clean_sql(raw_text, mode=MODE)

    # Keep all original fields from the input example.
    result = dict(example)

    # Map gold_sql_query -> gold_sql
    gold_sql_value = example.get("gold_sql_query", "")

    result.update({
        "instance_id": example.get("instance_id", ""),
        "question": example.get("question", ""),
        "gold_sql": gold_sql_value,
        "predicted_sql": predicted_sql,
        "latency_ms": round(float(latency_ms), 3),
        "num_tokens_generated": int(num_tokens_generated),
    })

    # Remove the old field if it exists
    if "gold_sql_query" in result:
        del result["gold_sql_query"]

    validate_output_record(result)
    results.append(result)

    if idx <= 3:
        print("=" * 80)
        print(f"Example {idx}")
        print("Question:", result["question"])
        print("Gold SQL:", result["gold_sql"])
        print("Predicted SQL:", result["predicted_sql"])
        print("Latency (ms):", result["latency_ms"])
        print("Generated tokens:", result["num_tokens_generated"])
    # if idx == 3:
    #   break

    if idx % 5 == 0 or idx == len(data):
        print(f"Done {idx}/{len(data)}")

output_filename = build_output_filename(
    dataset_name=DATASET_NAME,
    mode=MODE,
    train_mode=TRAIN_MODE,
    constraint_method=CONSTRAINT_METHOD,
)

output_path = os.path.join(OUTPUT_DIR, output_filename)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to:", output_path)

Example 1
Question: What is the title of the episode directed by Peter Markle and written by Jerry Stahl?
Gold SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Predicted SQL: 
Latency (ms): 56.384
Generated tokens: 1000
Example 2
Question: What's the total number of episodes with production code 2T6705?
Gold SQL: SELECT COUNT(Title) FROM table WHERE Production code = '2T6705'
Predicted SQL: 
Latency (ms): 62.685
Generated tokens: 1000
Example 3
Question: How many times was the GFR 2006 equal to 53.2?
Gold SQL: SELECT COUNT(Whites as % of Pop.) FROM table WHERE GFR 2006 = '53.2'
Predicted SQL: SELECT COUNT(*) FROM table WHERE GFR 2006 = '53.2'
Latency (ms): 1.376
Generated tokens: 22
Done 5/50
Done 10/50
Done 15/50
Done 20/50
Done 25/50
Done 30/50
Done 35/50
Done 40/50
Done 45/50
Done 50/50
Saved to: outputs/result_wikisql_thinking_zero_xgrammar.json


### new constrained

In [ ]:
from transformers import AutoConfig
# Choose from: "thinking", "non-thinking"
MODE = "non-thinking"
# MODE = "thinking"

# Choose from: "zero", "ft"
TRAIN_MODE = "zero"

# Choose from: "none", "outlines" or "xgrammar"
# CONSTRAINT_METHOD = "none"
CONSTRAINT_METHOD = "xgrammar"

MAX_NEW_TOKENS = 100

os.makedirs(OUTPUT_DIR, exist_ok=True)

generation_kwargs = build_generation_kwargs(
    mode=MODE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=MIN_P,
)

outlines_runner = None
if CONSTRAINT_METHOD == "outlines":
    outlines_runner = OutlinesRunner(model, tokenizer)

grammar_compiler = None
if CONSTRAINT_METHOD == "xgrammar":
    grammar_compiler = init_xgrammar(tokenizer, MODEL_NAME)

results = []

for idx, example in enumerate(data, start=1):
    prompt = build_qwen_prompt(tokenizer, example, MODE)

    if CONSTRAINT_METHOD == "outlines":
        regex_pattern = build_better_wikisql_regex()
        raw_text, num_tokens_generated, latency_ms = outlines_runner.generate_regex(
            prompt=prompt,
            regex_pattern=regex_pattern,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    elif CONSTRAINT_METHOD == "xgrammar":
        raw_text, num_tokens_generated, latency_ms = generate_xgrammar_new(
        model=model,
        tokenizer=tokenizer,
        device=model.device,
        prompt=prompt,
        example=example,
        grammar_compiler=grammar_compiler,
        generation_kwargs=generation_kwargs,
    )
    else:
        raw_text, num_tokens_generated, latency_ms = generate_plain(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            generation_kwargs=generation_kwargs,
        )

    predicted_sql = clean_sql(raw_text, mode=MODE)

    # Keep all original fields from the input example.
    result = dict(example)

    # Map gold_sql_query -> gold_sql
    gold_sql_value = example.get("gold_sql_query", "")

    result.update({
        "instance_id": example.get("instance_id", ""),
        "question": example.get("question", ""),
        "gold_sql": gold_sql_value,
        "predicted_sql": predicted_sql,
        "latency_ms": round(float(latency_ms), 3),
        "num_tokens_generated": int(num_tokens_generated),
    })

    # Remove the old field if it exists
    if "gold_sql_query" in result:
        del result["gold_sql_query"]

    validate_output_record(result)
    results.append(result)

    if idx <= 3:
        print("=" * 80)
        print(f"Example {idx}")
        print("Question:", result["question"])
        print("Gold SQL:", result["gold_sql"])
        print("Predicted SQL:", result["predicted_sql"])
        print("Latency (ms):", result["latency_ms"])
        print("Generated tokens:", result["num_tokens_generated"])
    # if idx == 3:
    #   break

    if idx % 5 == 0 or idx == len(data):
        print(f"Done {idx}/{len(data)}")

output_filename = build_output_filename(
    dataset_name=DATASET_NAME,
    mode=MODE,
    train_mode=TRAIN_MODE,
    constraint_method=CONSTRAINT_METHOD,
)

output_path = os.path.join(OUTPUT_DIR, output_filename)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to:", output_path)

Example 1
Question: What is the title of the episode directed by Peter Markle and written by Jerry Stahl?
Gold SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Predicted SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Latency (ms): 1.573
Generated tokens: 23
Example 2
Question: What's the total number of episodes with production code 2T6705?
Gold SQL: SELECT COUNT(Title) FROM table WHERE Production code = '2T6705'
Predicted SQL: SELECT COUNT(*) FROM table WHERE Production code = '2T6705'
Latency (ms): 1.11
Generated tokens: 18
Example 3
Question: How many times was the GFR 2006 equal to 53.2?
Gold SQL: SELECT COUNT(Whites as % of Pop.) FROM table WHERE GFR 2006 = '53.2'
Predicted SQL: SELECT COUNT(*) FROM table WHERE GFR 2006 = '53.2'
Latency (ms): 1.26
Generated tokens: 21
Done 5/50
Done 10/50
Done 15/50
Done 20/50
Done 25/50
Done 30/50
Done 35/50
Done 40/50
Done 45/50
Done 50/50
Saved to: out

In [ ]:
# Choose from: "thinking", "non-thinking"
# MODE = "thinking"
MODE = "thinking"

# Choose from: "zero", "ft"
TRAIN_MODE = "zero"

# Choose from: "none", "outlines"，"xgrammar
# CONSTRAINT_METHOD = "none"
CONSTRAINT_METHOD = "xgrammar"

MAX_NEW_TOKENS = 1000

os.makedirs(OUTPUT_DIR, exist_ok=True)

generation_kwargs = build_generation_kwargs(
    mode=MODE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=MIN_P,
)

outlines_runner = None
if CONSTRAINT_METHOD == "outlines":
    outlines_runner = OutlinesRunner(model, tokenizer)

grammar_compiler = None
if CONSTRAINT_METHOD == "xgrammar":
    grammar_compiler = init_xgrammar(tokenizer, MODEL_NAME)

results = []

for idx, example in enumerate(data, start=1):
    prompt = build_qwen_prompt(tokenizer, example, MODE)

    if CONSTRAINT_METHOD == "outlines":
        regex_pattern = build_better_wikisql_regex()
        raw_text, num_tokens_generated, latency_ms = outlines_runner.generate_regex(
            prompt=prompt,
            regex_pattern=regex_pattern,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    elif CONSTRAINT_METHOD == "xgrammar":
        raw_text, num_tokens_generated, latency_ms = generate_xgrammar_new(
        model=model,
        tokenizer=tokenizer,
        device=model.device,
        prompt=prompt,
        example=example,
        grammar_compiler=grammar_compiler,
        generation_kwargs=generation_kwargs,
    )
    else:
        raw_text, num_tokens_generated, latency_ms = generate_plain(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            generation_kwargs=generation_kwargs,
        )

    predicted_sql = clean_sql(raw_text, mode=MODE)

    # Keep all original fields from the input example.
    result = dict(example)

    # Map gold_sql_query -> gold_sql
    gold_sql_value = example.get("gold_sql_query", "")

    result.update({
        "instance_id": example.get("instance_id", ""),
        "question": example.get("question", ""),
        "gold_sql": gold_sql_value,
        "predicted_sql": predicted_sql,
        "latency_ms": round(float(latency_ms), 3),
        "num_tokens_generated": int(num_tokens_generated),
    })

    # Remove the old field if it exists
    if "gold_sql_query" in result:
        del result["gold_sql_query"]

    validate_output_record(result)
    results.append(result)

    if idx <= 3:
        print("=" * 80)
        print(f"Example {idx}")
        print("Question:", result["question"])
        print("Gold SQL:", result["gold_sql"])
        print("Predicted SQL:", result["predicted_sql"])
        print("Latency (ms):", result["latency_ms"])
        print("Generated tokens:", result["num_tokens_generated"])
    # if idx == 3:
    #   break

    if idx % 5 == 0 or idx == len(data):
        print(f"Done {idx}/{len(data)}")

output_filename = build_output_filename(
    dataset_name=DATASET_NAME,
    mode=MODE,
    train_mode=TRAIN_MODE,
    constraint_method=CONSTRAINT_METHOD,
)

output_path = os.path.join(OUTPUT_DIR, output_filename)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to:", output_path)

Example 1
Question: What is the title of the episode directed by Peter Markle and written by Jerry Stahl?
Gold SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Predicted SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl' LIMIT 10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000

### extend

In [ ]:
from transformers import AutoConfig
# Choose from: "thinking", "non-thinking"
MODE = "non-thinking"
# MODE = "thinking"

# Choose from: "zero", "ft"
TRAIN_MODE = "zero"

# Choose from: "none", "outlines" or "xgrammar"
# CONSTRAINT_METHOD = "none"
CONSTRAINT_METHOD = "xgrammar"

MAX_NEW_TOKENS = 100

os.makedirs(OUTPUT_DIR, exist_ok=True)

generation_kwargs = build_generation_kwargs(
    mode=MODE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=MIN_P,
)

outlines_runner = None
if CONSTRAINT_METHOD == "outlines":
    outlines_runner = OutlinesRunner(model, tokenizer)

grammar_compiler = None
if CONSTRAINT_METHOD == "xgrammar":
    grammar_compiler = init_xgrammar(tokenizer, MODEL_NAME)

results = []

for idx, example in enumerate(data, start=1):
    prompt = build_qwen_prompt(tokenizer, example, MODE)

    if CONSTRAINT_METHOD == "outlines":
        regex_pattern = build_better_wikisql_regex()
        raw_text, num_tokens_generated, latency_ms = outlines_runner.generate_regex(
            prompt=prompt,
            regex_pattern=regex_pattern,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    elif CONSTRAINT_METHOD == "xgrammar":
        raw_text, num_tokens_generated, latency_ms = generate_xgrammar_extend(
        model=model,
        tokenizer=tokenizer,
        device=model.device,
        prompt=prompt,
        example=example,
        grammar_compiler=grammar_compiler,
        generation_kwargs=generation_kwargs,
    )
    else:
        raw_text, num_tokens_generated, latency_ms = generate_plain(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            generation_kwargs=generation_kwargs,
        )

    predicted_sql = clean_sql(raw_text, mode=MODE)

    # Keep all original fields from the input example.
    result = dict(example)

    # Map gold_sql_query -> gold_sql
    gold_sql_value = example.get("gold_sql_query", "")

    result.update({
        "instance_id": example.get("instance_id", ""),
        "question": example.get("question", ""),
        "gold_sql": gold_sql_value,
        "predicted_sql": predicted_sql,
        "latency_ms": round(float(latency_ms), 3),
        "num_tokens_generated": int(num_tokens_generated),
    })

    # Remove the old field if it exists
    if "gold_sql_query" in result:
        del result["gold_sql_query"]

    validate_output_record(result)
    results.append(result)

    if idx <= 3:
        print("=" * 80)
        print(f"Example {idx}")
        print("Question:", result["question"])
        print("Gold SQL:", result["gold_sql"])
        print("Predicted SQL:", result["predicted_sql"])
        print("Latency (ms):", result["latency_ms"])
        print("Generated tokens:", result["num_tokens_generated"])
    # if idx == 3:
    #   break

    if idx % 5 == 0 or idx == len(data):
        print(f"Done {idx}/{len(data)}")

output_filename = build_output_filename(
    dataset_name=DATASET_NAME,
    mode=MODE,
    train_mode=TRAIN_MODE,
    constraint_method=CONSTRAINT_METHOD,
)

output_path = os.path.join(OUTPUT_DIR, output_filename)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to:", output_path)

Example 1
Question: What is the title of the episode directed by Peter Markle and written by Jerry Stahl?
Gold SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Predicted SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Latency (ms): 1.712
Generated tokens: 23
Example 2
Question: What's the total number of episodes with production code 2T6705?
Gold SQL: SELECT COUNT(Title) FROM table WHERE Production code = '2T6705'
Predicted SQL: SELECT COUNT ( * ) FROM table WHERE Production code = '2T6705'
Latency (ms): 1.643
Generated tokens: 20
Example 3
Question: How many times was the GFR 2006 equal to 53.2?
Gold SQL: SELECT COUNT(Whites as % of Pop.) FROM table WHERE GFR 2006 = '53.2'
Predicted SQL: SELECT COUNT ( * ) FROM table WHERE GFR 2006 = '53.2'
Latency (ms): 1.437
Generated tokens: 23
Done 5/50
Done 10/50
Done 15/50
Done 20/50
Done 25/50
Done 30/50
Done 35/50
Done 40/50
Done 45/50
Done 50/50
Saved

In [ ]:
# Choose from: "thinking", "non-thinking"
# MODE = "thinking"
MODE = "thinking"

# Choose from: "zero", "ft"
TRAIN_MODE = "zero"

# Choose from: "none", "outlines"，"xgrammar
# CONSTRAINT_METHOD = "none"
CONSTRAINT_METHOD = "xgrammar"

MAX_NEW_TOKENS = 1000

os.makedirs(OUTPUT_DIR, exist_ok=True)

generation_kwargs = build_generation_kwargs(
    mode=MODE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=MIN_P,
)

outlines_runner = None
if CONSTRAINT_METHOD == "outlines":
    outlines_runner = OutlinesRunner(model, tokenizer)

grammar_compiler = None
if CONSTRAINT_METHOD == "xgrammar":
    grammar_compiler = init_xgrammar(tokenizer, MODEL_NAME)

results = []

for idx, example in enumerate(data, start=1):
    prompt = build_qwen_prompt(tokenizer, example, MODE)

    if CONSTRAINT_METHOD == "outlines":
        regex_pattern = build_better_wikisql_regex()
        raw_text, num_tokens_generated, latency_ms = outlines_runner.generate_regex(
            prompt=prompt,
            regex_pattern=regex_pattern,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    elif CONSTRAINT_METHOD == "xgrammar":
        raw_text, num_tokens_generated, latency_ms = generate_xgrammar_extend(
        model=model,
        tokenizer=tokenizer,
        device=model.device,
        prompt=prompt,
        example=example,
        grammar_compiler=grammar_compiler,
        generation_kwargs=generation_kwargs,
    )
    else:
        raw_text, num_tokens_generated, latency_ms = generate_plain(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            generation_kwargs=generation_kwargs,
        )

    predicted_sql = clean_sql(raw_text, mode=MODE)

    # Keep all original fields from the input example.
    result = dict(example)

    # Map gold_sql_query -> gold_sql
    gold_sql_value = example.get("gold_sql_query", "")

    result.update({
        "instance_id": example.get("instance_id", ""),
        "question": example.get("question", ""),
        "gold_sql": gold_sql_value,
        "predicted_sql": predicted_sql,
        "latency_ms": round(float(latency_ms), 3),
        "num_tokens_generated": int(num_tokens_generated),
    })

    # Remove the old field if it exists
    if "gold_sql_query" in result:
        del result["gold_sql_query"]

    validate_output_record(result)
    results.append(result)

    if idx <= 3:
        print("=" * 80)
        print(f"Example {idx}")
        print("Question:", result["question"])
        print("Gold SQL:", result["gold_sql"])
        print("Predicted SQL:", result["predicted_sql"])
        print("Latency (ms):", result["latency_ms"])
        print("Generated tokens:", result["num_tokens_generated"])
    # if idx == 3:
    #   break

    if idx % 5 == 0 or idx == len(data):
        print(f"Done {idx}/{len(data)}")

output_filename = build_output_filename(
    dataset_name=DATASET_NAME,
    mode=MODE,
    train_mode=TRAIN_MODE,
    constraint_method=CONSTRAINT_METHOD,
)

output_path = os.path.join(OUTPUT_DIR, output_filename)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to:", output_path)

# simple evaluation

In [ ]:
import json
import sqlparse

def normalize_sql(sql):
    if sql is None:
        return ""
    return " ".join(sql.strip().rstrip(";").lower().split())

def is_valid_sql(sql):
    try:
        return len(sqlparse.parse(sql)) > 0
    except:
        return False


def evaluate_auto(file_path):
    total = 0
    exact_match = 0
    empty_pred = 0
    valid_sql = 0
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            data = json.load(f)
            if isinstance(data, dict):
                data = [data]
        except json.JSONDecodeError:
            f.seek(0)
            data = [json.loads(line) for line in f if line.strip()]

    for r in data:
        gold = r.get("gold_sql_query") or r.get("gold_sql") or ""
        pred = r.get("pred_sql") or r.get("predicted_sql") or ""

        total += 1

        if pred.strip() == "":
            empty_pred += 1

        if is_valid_sql(pred):
            valid_sql += 1

        if normalize_sql(gold) == normalize_sql(pred):
            exact_match += 1

    # -------- 统计 --------
    em_rate = exact_match / total if total > 0 else 0
    empty_rate = empty_pred / total if total > 0 else 0
    valid_rate = valid_sql / total if total > 0 else 0

    print("\n===== Evaluation Result =====")
    print(f"Total: {total}")
    print(f"Exact Match: {exact_match}/{total} = {em_rate:.4f}")
    print(f"Valid SQL: {valid_sql}/{total} = {valid_rate:.4f}")
    print(f"Empty Prediction: {empty_pred}/{total} = {empty_rate:.4f}")
    print("=============================\n")

    return {
        "total": total,
        "exact_match": exact_match,
        "exact_match_rate": em_rate,
        "valid_sql": valid_sql,
        "valid_sql_rate": valid_rate,
        "empty_pred": empty_pred,
        "empty_pred_rate": empty_rate,
    }

In [ ]:
result_path = "outputs/result_wikisql_non-thinking_zero_xgrammar.json"
evaluate_auto(result_path)


===== Evaluation Result =====
Total: 50
Exact Match: 12/50 = 0.2400
Valid SQL: 49/50 = 0.9800
Empty Prediction: 1/50 = 0.0200



{'total': 50,
 'exact_match': 12,
 'exact_match_rate': 0.24,
 'valid_sql': 49,
 'valid_sql_rate': 0.98,
 'empty_pred': 1,
 'empty_pred_rate': 0.02}

# Outlines

In [ ]:
# Choose from: "thinking", "non-thinking"
MODE = "non-thinking"
# MODE = "thinking"

# Choose from: "zero", "ft"
TRAIN_MODE = "zero"

# Choose from: "none", "outlines"
# CONSTRAINT_METHOD = "none"
CONSTRAINT_METHOD = "outlines"

MAX_NEW_TOKENS = 100

os.makedirs(OUTPUT_DIR, exist_ok=True)

generation_kwargs = build_generation_kwargs(
    mode=MODE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=MIN_P,
)

outlines_runner = None
if CONSTRAINT_METHOD == "outlines":
    outlines_runner = OutlinesRunner(model, tokenizer)

results = []

for idx, example in enumerate(data, start=1):
    prompt = build_qwen_prompt(tokenizer, example, MODE)

    if CONSTRAINT_METHOD == "outlines":
        regex_pattern = build_better_wikisql_regex()
        raw_text, num_tokens_generated, latency_ms = outlines_runner.generate_regex(
            prompt=prompt,
            regex_pattern=regex_pattern,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    else:
        raw_text, num_tokens_generated, latency_ms = generate_plain(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            generation_kwargs=generation_kwargs,
        )

    predicted_sql = clean_sql(raw_text, mode=MODE)

    # Keep all original fields from the input example.
    result = dict(example)

    # Map gold_sql_query -> gold_sql
    gold_sql_value = example.get("gold_sql_query", "")

    result.update({
        "instance_id": example.get("instance_id", ""),
        "question": example.get("question", ""),
        "gold_sql": gold_sql_value,
        "predicted_sql": predicted_sql,
        "latency_ms": round(float(latency_ms), 3),
        "num_tokens_generated": int(num_tokens_generated),
    })

    # Remove the old field if it exists
    if "gold_sql_query" in result:
        del result["gold_sql_query"]

    validate_output_record(result)
    results.append(result)

    if idx <= 3:
        print("=" * 80)
        print(f"Example {idx}")
        print("Question:", result["question"])
        print("Gold SQL:", result["gold_sql"])
        print("Predicted SQL:", result["predicted_sql"])
        print("Latency (ms):", result["latency_ms"])
        print("Generated tokens:", result["num_tokens_generated"])
    # if idx == 3:
    #   break

    if idx % 5 == 0 or idx == len(data):
        print(f"Done {idx}/{len(data)}")

output_filename = build_output_filename(
    dataset_name=DATASET_NAME,
    mode=MODE,
    train_mode=TRAIN_MODE,
    constraint_method=CONSTRAINT_METHOD,
)

output_path = os.path.join(OUTPUT_DIR, output_filename)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to:", output_path)

Example 1
Question: What is the title of the episode directed by Peter Markle and written by Jerry Stahl?
Gold SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Predicted SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Latency (ms): 4328.077
Generated tokens: 22
Example 2
Question: What's the total number of episodes with production code 2T6705?
Gold SQL: SELECT COUNT(Title) FROM table WHERE Production code = '2T6705'
Predicted SQL: SELECT SUM(U_S_VIEWERS) FROM table WHERE Production code = '2T6705'
Latency (ms): 3983.103
Generated tokens: 21
Example 3
Question: How many times was the GFR 2006 equal to 53.2?
Gold SQL: SELECT COUNT(Whites as % of Pop.) FROM table WHERE GFR 2006 = '53.2'
Predicted SQL: SELECT COUNT(*) FROM table WHERE GFR 2006 = '53.2'
Latency (ms): 4683.379
Generated tokens: 20
Done 5/50
Done 10/50
Done 15/50
Done 20/50
Done 25/50
Done 30/50
Done 35/50
Done 40/50
Done 45/50
Done 

In [ ]:
# Choose from: "thinking", "non-thinking"
# MODE = "non-thinking"
MODE = "thinking"

# Choose from: "zero", "ft"
TRAIN_MODE = "zero"

# Choose from: "none", "outlines"
# CONSTRAINT_METHOD = "none"
CONSTRAINT_METHOD = "outlines"

MAX_NEW_TOKENS = 1000

os.makedirs(OUTPUT_DIR, exist_ok=True)

generation_kwargs = build_generation_kwargs(
    mode=MODE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=MIN_P,
)

outlines_runner = None
if CONSTRAINT_METHOD == "outlines":
    outlines_runner = OutlinesRunner(model, tokenizer)

results = []

for idx, example in enumerate(data, start=1):
    prompt = build_qwen_prompt(tokenizer, example, MODE)

    if CONSTRAINT_METHOD == "outlines":
        regex_pattern = build_better_wikisql_regex()
        raw_text, num_tokens_generated, latency_ms = outlines_runner.generate_regex(
            prompt=prompt,
            regex_pattern=regex_pattern,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    else:
        raw_text, num_tokens_generated, latency_ms = generate_plain(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            generation_kwargs=generation_kwargs,
        )

    predicted_sql = clean_sql(raw_text, mode=MODE)

    # Keep all original fields from the input example.
    result = dict(example)

    # Map gold_sql_query -> gold_sql
    gold_sql_value = example.get("gold_sql_query", "")

    result.update({
        "instance_id": example.get("instance_id", ""),
        "question": example.get("question", ""),
        "gold_sql": gold_sql_value,
        "predicted_sql": predicted_sql,
        "latency_ms": round(float(latency_ms), 3),
        "num_tokens_generated": int(num_tokens_generated),
    })

    # Remove the old field if it exists
    if "gold_sql_query" in result:
        del result["gold_sql_query"]

    validate_output_record(result)
    results.append(result)

    if idx <= 3:
        print("=" * 80)
        print(f"Example {idx}")
        print("Question:", result["question"])
        print("Gold SQL:", result["gold_sql"])
        print("Predicted SQL:", result["predicted_sql"])
        print("Latency (ms):", result["latency_ms"])
        print("Generated tokens:", result["num_tokens_generated"])
    # if idx == 3:
    #   break

    if idx % 5 == 0 or idx == len(data):
        print(f"Done {idx}/{len(data)}")

output_filename = build_output_filename(
    dataset_name=DATASET_NAME,
    mode=MODE,
    train_mode=TRAIN_MODE,
    constraint_method=CONSTRAINT_METHOD,
)

output_path = os.path.join(OUTPUT_DIR, output_filename)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to:", output_path)

Example 1
Question: What is the title of the episode directed by Peter Markle and written by Jerry Stahl?
Gold SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Predicted SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Latency (ms): 3924.76
Generated tokens: 22
Example 2
Question: What's the total number of episodes with production code 2T6705?
Gold SQL: SELECT COUNT(Title) FROM table WHERE Production code = '2T6705'
Predicted SQL: SELECT COUNT(*) FROM table WHERE Production code = '2T6705'
Latency (ms): 3953.967
Generated tokens: 17
Example 3
Question: How many times was the GFR 2006 equal to 53.2?
Gold SQL: SELECT COUNT(Whites as % of Pop.) FROM table WHERE GFR 2006 = '53.2'
Predicted SQL: SELECT COUNT(*) FROM table WHERE GFR 2006 = '53.2'
Latency (ms): 3725.213
Generated tokens: 20
Done 5/50
Done 10/50
Done 15/50
Done 20/50
Done 25/50
Done 30/50
Done 35/50
Done 40/50
Done 45/50
Done 50/50
Sav

In [ ]:
# Choose from: "thinking", "non-thinking"
MODE = "non-thinking"
# MODE = "thinking"

# Choose from: "zero", "ft"
TRAIN_MODE = "zero"

# Choose from: "none", "outlines"
CONSTRAINT_METHOD = "none"
# CONSTRAINT_METHOD = "outlines"

MAX_NEW_TOKENS = 100

os.makedirs(OUTPUT_DIR, exist_ok=True)

generation_kwargs = build_generation_kwargs(
    mode=MODE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=MIN_P,
)

outlines_runner = None
if CONSTRAINT_METHOD == "outlines":
    outlines_runner = OutlinesRunner(model, tokenizer)

results = []

for idx, example in enumerate(data, start=1):
    prompt = build_qwen_prompt(tokenizer, example, MODE)

    if CONSTRAINT_METHOD == "outlines":
        regex_pattern = build_better_wikisql_regex()
        raw_text, num_tokens_generated, latency_ms = outlines_runner.generate_regex(
            prompt=prompt,
            regex_pattern=regex_pattern,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    else:
        raw_text, num_tokens_generated, latency_ms = generate_plain(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            generation_kwargs=generation_kwargs,
        )

    predicted_sql = clean_sql(raw_text, mode=MODE)

    # Keep all original fields from the input example.
    result = dict(example)

    # Map gold_sql_query -> gold_sql
    gold_sql_value = example.get("gold_sql_query", "")

    result.update({
        "instance_id": example.get("instance_id", ""),
        "question": example.get("question", ""),
        "gold_sql": gold_sql_value,
        "predicted_sql": predicted_sql,
        "latency_ms": round(float(latency_ms), 3),
        "num_tokens_generated": int(num_tokens_generated),
    })

    # Remove the old field if it exists
    if "gold_sql_query" in result:
        del result["gold_sql_query"]

    validate_output_record(result)
    results.append(result)

    if idx <= 3:
        print("=" * 80)
        print(f"Example {idx}")
        print("Question:", result["question"])
        print("Gold SQL:", result["gold_sql"])
        print("Predicted SQL:", result["predicted_sql"])
        print("Latency (ms):", result["latency_ms"])
        print("Generated tokens:", result["num_tokens_generated"])
    # if idx == 3:
    #   break

    if idx % 5 == 0 or idx == len(data):
        print(f"Done {idx}/{len(data)}")

output_filename = build_output_filename(
    dataset_name=DATASET_NAME,
    mode=MODE,
    train_mode=TRAIN_MODE,
    constraint_method=CONSTRAINT_METHOD,
)

output_path = os.path.join(OUTPUT_DIR, output_filename)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to:", output_path)

Example 1
Question: What is the title of the episode directed by Peter Markle and written by Jerry Stahl?
Gold SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Predicted SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Latency (ms): 1181.511
Generated tokens: 23
Example 2
Question: What's the total number of episodes with production code 2T6705?
Gold SQL: SELECT COUNT(Title) FROM table WHERE Production code = '2T6705'
Predicted SQL: SELECT COUNT(*) FROM table WHERE Production code = '2T6705'
Latency (ms): 919.331
Generated tokens: 18
Example 3
Question: How many times was the GFR 2006 equal to 53.2?
Gold SQL: SELECT COUNT(Whites as % of Pop.) FROM table WHERE GFR 2006 = '53.2'
Predicted SQL: SELECT COUNT(*) FROM table WHERE GFR 2006 = '53.2'
Latency (ms): 1520.305
Generated tokens: 21
Done 5/50
Done 10/50
Done 15/50
Done 20/50
Done 25/50
Done 30/50
Done 35/50
Done 40/50
Done 45/50
Done 50/50
Sav

In [ ]:
# Choose from: "thinking", "non-thinking"
# MODE = "non-thinking"
MODE = "thinking"

# Choose from: "zero", "ft"
TRAIN_MODE = "zero"

# Choose from: "none", "outlines"
CONSTRAINT_METHOD = "none"
# CONSTRAINT_METHOD = "outlines"

MAX_NEW_TOKENS = 1000

os.makedirs(OUTPUT_DIR, exist_ok=True)

generation_kwargs = build_generation_kwargs(
    mode=MODE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=MIN_P,
)

outlines_runner = None
if CONSTRAINT_METHOD == "outlines":
    outlines_runner = OutlinesRunner(model, tokenizer)

results = []

for idx, example in enumerate(data, start=1):
    prompt = build_qwen_prompt(tokenizer, example, MODE)

    if CONSTRAINT_METHOD == "outlines":
        regex_pattern = build_better_wikisql_regex()
        raw_text, num_tokens_generated, latency_ms = outlines_runner.generate_regex(
            prompt=prompt,
            regex_pattern=regex_pattern,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    else:
        raw_text, num_tokens_generated, latency_ms = generate_plain(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            generation_kwargs=generation_kwargs,
        )

    predicted_sql = clean_sql(raw_text, mode=MODE)

    # Keep all original fields from the input example.
    result = dict(example)

    # Map gold_sql_query -> gold_sql
    gold_sql_value = example.get("gold_sql_query", "")

    result.update({
        "instance_id": example.get("instance_id", ""),
        "question": example.get("question", ""),
        "gold_sql": gold_sql_value,
        "predicted_sql": predicted_sql,
        "latency_ms": round(float(latency_ms), 3),
        "num_tokens_generated": int(num_tokens_generated),
    })

    # Remove the old field if it exists
    if "gold_sql_query" in result:
        del result["gold_sql_query"]

    validate_output_record(result)
    results.append(result)

    if idx <= 3:
        print("=" * 80)
        print(f"Example {idx}")
        print("Question:", result["question"])
        print("Gold SQL:", result["gold_sql"])
        print("Predicted SQL:", result["predicted_sql"])
        print("Latency (ms):", result["latency_ms"])
        print("Generated tokens:", result["num_tokens_generated"])
    # if idx == 3:
    #   break

    if idx % 5 == 0 or idx == len(data):
        print(f"Done {idx}/{len(data)}")

output_filename = build_output_filename(
    dataset_name=DATASET_NAME,
    mode=MODE,
    train_mode=TRAIN_MODE,
    constraint_method=CONSTRAINT_METHOD,
)

output_path = os.path.join(OUTPUT_DIR, output_filename)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to:", output_path)

Example 1
Question: What is the title of the episode directed by Peter Markle and written by Jerry Stahl?
Gold SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Predicted SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Latency (ms): 16041.275
Generated tokens: 302
Example 2
Question: What's the total number of episodes with production code 2T6705?
Gold SQL: SELECT COUNT(Title) FROM table WHERE Production code = '2T6705'
Predicted SQL: SELECT COUNT(*) FROM table WHERE Production code = '2T6705'
Latency (ms): 14152.452
Generated tokens: 276
Example 3
Question: How many times was the GFR 2006 equal to 53.2?
Gold SQL: SELECT COUNT(Whites as % of Pop.) FROM table WHERE GFR 2006 = '53.2'
Predicted SQL: SELECT COUNT(*) FROM table WHERE GFR 2006 = '53.2'
Latency (ms): 42526.63
Generated tokens: 826
Done 5/50
Done 10/50
Done 15/50
Done 20/50
Done 25/50
Done 30/50
Done 35/50
Done 40/50
Done 45/50
Done 50/